# LEANN Drive Indexing Notebook

Use this notebook to mount a drive folder, build a LEANN index from your documents, and run semantic search or chat against the indexed content. Update the configuration cells with your own paths and API keys before running.


## 1) Install LEANN

The `[documents]` extra pulls in helpful parsers for PDFs, Word files, and spreadsheets. If you're running in Colab, remove the leading `!` before `pip` when converting this notebook to a script.


In [ ]:
!pip install -q 'leann[documents]'

## 2) Configure API keys and paths

Fill in your OpenAI-compatible API key (or adjust the `llm_config` later for other providers) and point `DATA_DIR` to the folder in your drive that holds the documents you want to index. `INDEX_PATH` determines where the LEANN index files will be written.


In [ ]:
import os
from pathlib import Path

# Required for chat generation; set to your provider's key.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "<your-openai-key>")
# Uncomment and customize if you use an OpenAI-compatible endpoint
# os.environ["OPENAI_BASE_URL"] = "https://api.your-provider.com/v1"

# Update these paths for your environment
DATA_DIR = Path("/content/drive/MyDrive/your-documents").expanduser()  # folder with your files
INDEX_PATH = Path("/content/drive/MyDrive/leann_indexes/drive_demo.leann").expanduser()
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)


## 3) (Optional) Mount Google Drive

Run this cell only if you're using Google Colab and your files live in Drive. If your data is already available on the local filesystem, skip this step.


In [ ]:
# Uncomment when running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')


## 4) Load documents from your drive folder

`SimpleDirectoryReader` recursively walks `DATA_DIR` and extracts text from common formats (TXT, Markdown, PDF, DOCX, XLSX, CSV). Adjust `required_exts` to narrow or widen the set of files you want to ingest.


In [ ]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(
    input_dir=str(DATA_DIR),
    recursive=True,
    required_exts=[".txt", ".md", ".pdf", ".docx", ".xlsx", ".xls", ".csv"],
)
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")


## 5) Build a LEANN index

Use the high-level `LeannBuilder` to add each document's text as a passage. You can attach per-passage metadata (such as the original file path) to help with filtering or attribution during search.


In [ ]:
from leann import LeannBuilder

# Choose an embedding model; BGE and Contriever are strong defaults.
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

builder = LeannBuilder(
    backend_name="hnsw",  # or "diskann" if you installed the optional backend
    embedding_model=EMBEDDING_MODEL,
)

for idx, doc in enumerate(documents):
    # doc.metadata may contain a file path or other structured data depending on the reader
    source_path = doc.metadata.get("file_path") or doc.metadata.get("file_name") or f"doc-{idx}"
    builder.add_text(doc.text, metadata={"source": source_path})

builder.build_index(str(INDEX_PATH))
print(f"Index built at {INDEX_PATH}")


## 6) Run semantic search

`LeannSearcher` loads the saved index and retrieves the most relevant passages for your query.


In [ ]:
from leann import LeannSearcher

searcher = LeannSearcher(str(INDEX_PATH))
results = searcher.search("What are the main takeaways in my documents?", top_k=3)

for hit in results:
    print(f"Score: {hit.score:.4f} | Source: {hit.metadata.get('source')}")
    print(hit.text[:400].strip())
    print("-" * 80)


## 7) Chat with your drive content

Configure an LLM provider (OpenAI-compatible by default) and ask grounded questions. The `top_k` parameter controls how many passages are fed into the model as context.


In [ ]:
from leann import LeannChat

llm_config = {
    "type": "openai",  # switch to "hf" for HuggingFace or "ollama" for local models
    "model": "gpt-4o-mini",  # replace with your chosen model
}

chat = LeannChat(str(INDEX_PATH), llm_config=llm_config)
response = chat.ask("Summarize the most important points in these files", top_k=5)
print(response)
